<a href="https://colab.research.google.com/github/parthsharma5575/Ai-Ml-GenAi/blob/main/ParthSharma_Assignment_Day3_Day4_15_Questions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment: Pre-training, Fine-tuning, RLHF & Context Window

Answer all 15 questions.

## Question 1

**Differentiate Pre-training and Fine-tuning with examples.**

**Answer:**

Pre-training is the first stage where a model learns general language patterns from a very large dataset. Fine-tuning is the second stage where the same model is adapted to a specific task or domain using a smaller dataset.

Example -> a model is pre-trained on internet text, then fine-tuned for customer support or medical question answering

## Question 2

**Explain self-supervised learning used in LLM pre-training.**

**Answer:**

Self-supervised learning means the model creates its own labels from raw text. For example, it learns to predict the next word or missing word from the surrounding words.
This helps the model learn language patterns without manual labeling.

## Question 3

**Why is fine-tuning cheaper than pre-training?**

**Answer:**

Fine-tuning is cheaper because it starts from an already trained base model. It needs less data, less computing power, and less time than training from scratch.
Pre-training is expensive because it uses massive datasets and long training runs.

## Question 4

**Differentiate Instruction Tuning and Fine-tuning.**

**Answer:**

Fine-tuning is a broad process of adapting a model for any target task. Instruction tuning is a special type of fine-tuning where the model learns to follow natural-language instructions better.
Example: instruction tuning helps a model respond well to prompts like “summarize this text” or “write a polite email”.

## Question 5

**What is RLHF? Explain all three stages.**

**Answer:**

RLHF means Reinforcement Learning from Human Feedback. It is used to make model outputs more helpful and aligned with human preferences.

The three stages are: pre-training a base model, training a reward model using human preferences, and then optimizing the model with reinforcement learning.

## Question 6

**Explain the role of human feedback in RLHF.**

**Answer:**

Human feedback tells the model which answers are better, safer, and more useful. Humans compare different outputs and help the system learn preferred behavior.
This is important because fluent answers are not always correct or safe.

## Question 7

**What is a context window? Why is it important?**

**Answer:**

A context window is the maximum amount of text a model can look at at one time. It includes the input prompt, previous conversation, and sometimes the output.

It is important because it decides how much information the model can use while answering.


## Question 8

**A model has an 8K context window. What does it mean?**

**Answer:**

An 8K context window means the model can handle about 8,000 tokens in one context. Tokens are pieces of text, not exactly words.
So the model can read and respond using roughly 8,000 tokens of combined input and output.

## Question 9

**Compare 8K, 32K and 128K context windows with use cases.**

**Answer:**

8K is suitable for short chats, small documents, and simple Q&A. 32K is better for long reports, longer articles, and detailed analysis. 128K is useful for large documents, long codebases, and multi-document reasoning.
As the window grows, cost and compute usage usually increase too.

## Question 10

**Write Python code to load a pretrained Hugging Face model and perform inference.**

**Answer:**

SmolLM2 HuggingFaceTB/SmolLM2-1.7B-Instruct 1.7B

In [1]:
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch


In [3]:
model="HuggingFaceTB/SmolLM2-1.7B-Instruct"
tokenizer=AutoTokenizer.from_pretrained(model)
model=AutoModelForCausalLM.from_pretrained(model)

Loading weights:   0%|          | 0/218 [00:00<?, ?it/s]

In [4]:
inputs=tokenizer("Hello,how are you , Tell me about yourself!",return_tensors="pt")
output=model.generate(**inputs,max_new_tokens=40)
output=tokenizer.decode(output[0],skip_special_tokens=True)
print(output)

Hello,how are you , Tell me about yourself!

I am a 25-year-old software engineer with a passion for learning new technologies and improving my skills. I have been working in the tech industry for the past three years,


## Question 11

**Write code to fine-tune a sentiment model on a custom dataset (outline acceptable).**

**Answer:**

In [5]:
!pip install transformers datasets accelerate evaluate

In [7]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer,DataCollatorWithPadding
from datasets import load_dataset

dataset = load_dataset("stanfordnlp/imdb")  # updated repo path
small_train_dataset = dataset["train"].shuffle(seed=42).select(range(1000))
small_eval_dataset = dataset["test"].shuffle(seed=42).select(range(1000))

checkpoint="bert-base-cased"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize_function(batch):
    return tokenizer(batch["text"], truncation=True)

tokenized_train = small_train_dataset.map(tokenize_function, batched=True)
tokenized_test = small_eval_dataset.map(tokenize_function, batched=True)

tokenized_train = tokenized_train.remove_columns(["text"])
tokenized_test = tokenized_test.remove_columns(["text"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

model = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=2
)
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator,
)

trainer.train()

results = trainer.evaluate()
print(results)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss
1,0.645685,0.467946
2,0.300383,0.341042
3,0.151710,0.427427


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training Loss,Validation Loss,Epoch
0.151710,0.427427,3


{'eval_loss': 0.42742717266082764}


## Question 12

**Debug: Why might an LLM forget earlier parts of a long conversation?**

**Answer:**

An LLM may forget earlier parts because the conversation becomes longer than its context window. When that happens, older text may be removed or not used fully.

It can also happen when the prompt has too much unrelated information, making earlier details harder to use.

## Question 13

**Case Study: Choose a model for legal document analysis and justify using context window and fine-tuning.**

**Answer:**

For legal document analysis, a model with a large context window is best, such as 32K or 128K. Legal files are long and often contain many clauses, so a bigger context helps.

Fine-tuning on legal text is also useful because it improves understanding of legal language, terminology, and document structure.

## Question 14

**Calculate approximate token usage for a 25-page document and discuss whether a 16K context window is sufficient.**

**Answer:**

A 25-page document is often around 6,000 to 10,000 tokens, depending on formatting and text density. A 16K context window is usually enough for one such document.

However, if you add long prompts, chat history, or multiple documents, 16K may become insufficient.

## Question 15

**Mini Project: Build a domain-specific chatbot using a pretrained model. Describe dataset, fine-tuning strategy, evaluation metrics, and deployment.**

**Answer:**

Import Libraries

In [8]:
!pip -q install transformers datasets sentencepiece accelerate

In [9]:
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    TrainingArguments,
    Trainer
)

Create sample data

In [10]:
data = {
    "question": [
        "How can I track my order?",
        "How do I cancel my order?",
        "What payment methods are accepted?",
        "Can I return a product?",
        "How long does shipping take?",
        "How do I reset my password?",
        "Do you offer cash on delivery?",
        "Where is my refund?",
        "Can I change my delivery address?",
        "How can I contact customer support?"
    ],

    "answer": [
        "You can track your order from the Orders section using your tracking ID.",
        "Go to My Orders and select Cancel Order before it is shipped.",
        "We accept credit cards, debit cards, UPI and net banking.",
        "Yes. Products can be returned within seven days of delivery.",
        "Shipping usually takes three to five business days.",
        "Click Forgot Password on the login page and follow the instructions.",
        "Yes, cash on delivery is available in selected locations.",
        "Refunds are processed within five to seven working days.",
        "Yes, you can change the address before the order is dispatched.",
        "You can contact us through email or our toll-free customer support number."
    ]
}

df = pd.DataFrame(data)

dataset = Dataset.from_pandas(df)

dataset

Dataset({
    features: ['question', 'answer'],
    num_rows: 10
})

Load tokenizer and model

In [11]:
checkpoint = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)

model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint)

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [12]:
def preprocess(example):

    model_input = tokenizer(
        example["question"],
        max_length=64,
        truncation=True
    )

    labels = tokenizer(
        text_target=example["answer"],
        max_length=64,
        truncation=True
    )

    model_input["labels"] = labels["input_ids"]

    return model_input

tokenized_dataset = dataset.map(preprocess)

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

Tokenize data

data collator


In [13]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

Fine-Tuning setup

In [14]:
training_args = TrainingArguments(

    output_dir="./chatbot",

    per_device_train_batch_size=2,

    num_train_epochs=20,

    learning_rate=2e-4,

    logging_steps=1,

    save_strategy="no",

    report_to="none"
)

Train the model

In [15]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=tokenized_dataset,

    data_collator=data_collator
)

In [16]:
trainer.train()

Step,Training Loss
1,3.049682
2,3.061378
3,2.632797
4,2.717220
5,2.607768
6,1.646198
7,2.297876
8,2.401911
9,2.139083
10,2.288118


TrainOutput(global_step=100, training_loss=0.768908905684948, metrics={'train_runtime': 27.8469, 'train_samples_per_second': 7.182, 'train_steps_per_second': 3.591, 'total_flos': 576551018496.0, 'train_loss': 0.768908905684948, 'epoch': 20.0})

Save the model

In [17]:
model.save_pretrained("chatbot_model")

tokenizer.save_pretrained("chatbot_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('chatbot_model/tokenizer_config.json', 'chatbot_model/tokenizer.json')

Load saved model

In [18]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("chatbot_model")

model = AutoModelForSeq2SeqLM.from_pretrained("chatbot_model")

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Chat with bot

In [19]:
question = "How can I track my order?"

inputs = tokenizer(
    question,
    return_tensors="pt"
)

outputs = model.generate(
    **inputs,
    max_new_tokens=40
)

print("User :", question)

print("Bot  :", tokenizer.decode(outputs[0], skip_special_tokens=True))

User : How can I track my order?
Bot  : You can track your order from the Orders section using your tracking ID.


In [20]:
question = "How do I get my refund?"

inputs = tokenizer(
    question,
    return_tensors="pt"
)

outputs = model.generate(
    **inputs,
    max_new_tokens=12
)

print("User :", question)

print("Bot  :", tokenizer.decode(outputs[0], skip_special_tokens=True))

User : How do I get my refund?
Bot  : You can get your refund from the seller through PayPal.
